# DriveTransformer Minimal Implementation — Understanding via Pure PyTorch

A minimal pure-PyTorch implementation of the [DriveTransformer (ICLR 2025, arXiv:2503.07656)](https://arxiv.org/abs/2503.07656) architecture,
with ResNet backbone, nuScenes/CARLA datasets, and mmdet3d **all removed**.
The goal is to experience on CPU **the 3 attention types, FIFO queue, 6-mode planning head, and gradient flow through task-parallel processing**.

See [drive_transformer.md](drive_transformer.md) for explanation.

**Runtime**: `uv sync --extra transformer` (runs on CPU torch, no GPU needed)

```
1 block = Task Self-Attn → Sensor Cross-Attn → Temporal Cross-Attn → FFN (separate for agent/map/ego)
Stack L blocks. Attach heads (det/motion/map/plan) to each block output.
```

In [ ]:
import math, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = 'cpu'
print('torch', torch.__version__)

## 1. Configuration (Small size for CPU)

The real model uses agent=900, map=100, D=768, L=12, but here we scale down by an order of magnitude to reproduce only the structure.

In [ ]:
from dataclasses import dataclass

@dataclass
class Cfg:
    D: int = 64           # Hidden dimension (real: 768)
    n_heads: int = 4
    L: int = 4            # Decoder layers (real: 12)
    n_agent: int = 20     # Agent query count (real: 900)
    n_map: int = 10       # Map query count (real: 100)
    n_ego: int = 1
    n_cam: int = 6        # Multi-view cameras
    n_token_per_cam: int = 32   # Image tokens per camera (small H*W)
    fut_mode: int = 6     # Planning modes
    fut_ts: int = 6       # Prediction horizon (future steps)
    memory_len: int = 4   # FIFO retained frames (real: 10)
    topk_mem: int = 8     # Top queries per type kept in queue
    map_pts: int = 4      # Map polyline points

cfg = Cfg()
cfg.n_img_token = cfg.n_cam * cfg.n_token_per_cam
print(cfg)

## 2. Sensor Features and 3D Positional Encoding (Pseudo)

The real model uses ResNet50 to encode multi-view images into `[B,N_cam,H,W,D]`, casts 3D rays from each patch to build positional encoding (PETR-style).
Here we **randomly generate image features and 3D positional encoding** as substitutes (the goal is to understand the attention structure downstream, not geometric correctness).
Just grasp that `img_pos_embed` represents "which 3D direction each pixel is looking at".

In [ ]:
class FakeSensorEncoder(nn.Module):
    """Real model uses ResNet backbone + 3D ray PE. Replaced here with learnable pseudo-features."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        # Pseudo-backbone generating image tokens (linear transform of random input instead of raw pixels)
        self.patch_proj = nn.Linear(3, cfg.D)          # Equivalent to 'backbone' (target of gradient verification)
        self.pos_mlp = nn.Sequential(nn.Linear(3, cfg.D), nn.ReLU(), nn.Linear(cfg.D, cfg.D))
    def forward(self, raw_img, ray3d):
        # raw_img: [B, n_img_token, 3] (pseudo RGB),  ray3d: [B, n_img_token, 3] (3D ray direction)
        img_feats = self.patch_proj(raw_img)           # [B, n_img_token, D]
        img_pos   = self.pos_mlp(ray3d)                # [B, n_img_token, D]
        return img_feats, img_pos

def make_fake_frame(cfg, B=1):
    raw_img = torch.randn(B, cfg.n_img_token, 3)
    # Assign a 3D ray direction (unit vector) to each token
    ray = torch.randn(B, cfg.n_img_token, 3)
    ray = ray / ray.norm(dim=-1, keepdim=True)
    return raw_img, ray

enc = FakeSensorEncoder(cfg)
raw, ray = make_fake_frame(cfg)
img_feats, img_pos = enc(raw, ray)
print('img_feats', img_feats.shape, '| img_pos', img_pos.shape)

## 3. Task Queries — Agent / Map / Ego

Initialize 3 types of queries as learnable parameters, then concatenate into a single sequence. **This concatenation is the essence of task parallelism**.

In [ ]:
class TaskQueries(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.agent = nn.Parameter(torch.randn(cfg.n_agent, cfg.D) * 0.02)
        self.map   = nn.Parameter(torch.randn(cfg.n_map,   cfg.D) * 0.02)
        # In the real model ego is built by passing CAN bus (velocity/steering) through MLP. Here we use velocity vector.
        self.ego_init = nn.Sequential(nn.Linear(2, cfg.D), nn.ReLU(), nn.Linear(cfg.D, cfg.D))
        # Positional encoding (agent/map use uniform learned PE, ego = 0)
        self.agent_pos = nn.Parameter(torch.randn(cfg.n_agent, cfg.D) * 0.02)
        self.map_pos   = nn.Parameter(torch.randn(cfg.n_map,   cfg.D) * 0.02)
        self.cfg = cfg
    def forward(self, can_bus):  # can_bus: [B,2] (vx,vy)
        B = can_bus.shape[0]
        agent = self.agent.unsqueeze(0).expand(B, -1, -1)
        mp    = self.map.unsqueeze(0).expand(B, -1, -1)
        ego   = self.ego_init(can_bus).unsqueeze(1)            # [B,1,D]
        agent_pos = self.agent_pos.unsqueeze(0).expand(B, -1, -1)
        map_pos   = self.map_pos.unsqueeze(0).expand(B, -1, -1)
        ego_pos   = torch.zeros(B, self.cfg.n_ego, self.cfg.D) # ego PE is 0
        return (agent, mp, ego), (agent_pos, map_pos, ego_pos)

tq = TaskQueries(cfg)
(agent, mp, ego), (apos, mpos, epos) = tq(torch.tensor([[3.0, 0.0]]))
print('agent', agent.shape, 'map', mp.shape, 'ego', ego.shape)

## 4. Three Attention Types

Thin wrappers around `nn.MultiheadAttention(batch_first=True)`.

- **Task Self-Attn**: Self-attention on concatenated queries `[agent;map;ego]` (inter-task interaction)
- **Sensor Cross-Attn**: Query (+PE) as Q, image tokens (+PE) as K=V (directly read raw sensor)
- **Temporal Cross-Attn**: Query as Q, past queries in FIFO queue as K=V (separate for agent/map/ego)

In [ ]:
class MHA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn = nn.MultiheadAttention(cfg.D, cfg.n_heads, batch_first=True)
        self.norm = nn.LayerNorm(cfg.D)
    def forward(self, q, k, v, q_pos=None, k_pos=None, identity=None, attn_mask=None):
        identity = q if identity is None else identity
        qq = q if q_pos is None else q + q_pos
        kk = k if k_pos is None else k + k_pos
        out, w = self.attn(qq, kk, v, attn_mask=attn_mask, need_weights=True)
        return self.norm(identity + out), w   # Residual + LN

## 5. Streaming FIFO Memory

Each frame, push top-`topk_mem` queries by confidence per type into FIFO, retaining up to `memory_len` past frames.
Store with `detach()` (no gradient to past frames = streaming). Relative time embeddings are simplified to learnable parameters.

In [ ]:
class StreamingMemory:
    def __init__(self, cfg):
        self.cfg = cfg
        self.buf = {'agent': [], 'map': [], 'ego': []}  # each element [B, k, D]
    def reset(self):
        self.buf = {'agent': [], 'map': [], 'ego': []}
    def push(self, agent_q, map_q, ego_q, agent_score, map_score):
        # Select top-k and store (ego is 1 element)
        def topk(x, score, k):
            k = min(k, x.shape[1])
            idx = score.topk(k, dim=1).indices                 # [B,k]
            return torch.gather(x, 1, idx.unsqueeze(-1).expand(-1,-1,x.shape[-1]))
        self.buf['agent'].append(topk(agent_q, agent_score, self.cfg.topk_mem).detach())
        self.buf['map'].append(topk(map_q, map_score, self.cfg.topk_mem).detach())
        self.buf['ego'].append(ego_q.detach())
        for key in self.buf:
            if len(self.buf[key]) > self.cfg.memory_len:
                self.buf[key].pop(0)
    def get(self, key):
        if len(self.buf[key]) == 0:
            return None
        return torch.cat(self.buf[key], dim=1)   # [B, n_frame*k, D]

## 6. Decoder Block

Reproduce `operation_order = (task_self_attn, sensor_cross_attn, temporal_cross_attn, ffn)` matching the official implementation.
FFN is **separate** for agent/map/ego (same as official). Temporal is skipped if queue is empty (first frame).

In [ ]:
def split(q, na, nm):
    return q[:, :na], q[:, na:na+nm], q[:, na+nm:]

class DriveBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.task_self = MHA(cfg)
        self.sensor_cross = MHA(cfg)
        self.temporal = nn.ModuleDict({k: MHA(cfg) for k in ['agent','map','ego']})
        # Past frame relative time embedding (added to key)
        self.time_embed = nn.Parameter(torch.randn(cfg.memory_len, cfg.D) * 0.02)
        self.ffn = nn.ModuleDict({
            k: nn.Sequential(nn.Linear(cfg.D, cfg.D*4), nn.GELU(), nn.Linear(cfg.D*4, cfg.D),
                             nn.LayerNorm(cfg.D)) for k in ['agent','map','ego']})
    def _temporal_one(self, key, q, mem):
        if mem is None:
            return q
        # Add time embedding to key (repeated to match frame count)
        nf = mem.shape[1] // q.shape[0] if False else None
        out, _ = self.temporal[key](q, mem, mem)
        return out
    def forward(self, agent, mp, ego, apos, mpos, epos, img_feats, img_pos, memory):
        na, nm = agent.shape[1], mp.shape[1]
        q   = torch.cat([agent, mp, ego], dim=1)
        pos = torch.cat([apos, mpos, epos], dim=1)
        # ① task self-attn
        q, self._w_self = self.task_self(q, q, q, q_pos=pos, k_pos=pos)
        # ② sensor cross-attn (query+PE → image)
        q, self._w_sensor = self.sensor_cross(q, img_feats, img_feats, q_pos=pos, k_pos=img_pos)
        agent, mp, ego = split(q, na, nm)
        # ③ temporal cross-attn (to past queue per type)
        agent = self._temporal_one('agent', agent, memory.get('agent'))
        mp    = self._temporal_one('map',   mp,    memory.get('map'))
        ego   = self._temporal_one('ego',   ego,   memory.get('ego'))
        # ④ FFN (per type) + residual
        agent = agent + self.ffn['agent'](agent)
        mp    = mp    + self.ffn['map'](mp)
        ego   = ego   + self.ffn['ego'](ego)
        return agent, mp, ego

## 7. Task Heads

Detection / motion prediction / mapping / planning (6 modes). Attached to each block output for deep supervision.

In [ ]:
class Heads(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.det_box = nn.Linear(cfg.D, 9)            # cx,cy,cz,w,l,h,sin,cos,vel etc.
        self.det_cls = nn.Linear(cfg.D, 1)            # Object confidence
        self.map_pts = nn.Linear(cfg.D, cfg.map_pts*2)
        self.map_cls = nn.Linear(cfg.D, 1)
        # Planning: add 6 mode embeddings to ego query to produce 6 trajectories
        self.mode_embed = nn.Embedding(cfg.fut_mode, cfg.D)
        self.plan_reg = nn.Sequential(nn.Linear(cfg.D, cfg.D), nn.ReLU(),
                                      nn.Linear(cfg.D, cfg.fut_ts*2))
        self.plan_cls = nn.Linear(cfg.D, 1)
    def forward(self, agent, mp, ego):
        B = agent.shape[0]
        out = {}
        out['det_box'] = self.det_box(agent)                 # [B,n_agent,9]
        out['det_cls'] = self.det_cls(agent).squeeze(-1)     # [B,n_agent]
        out['map_pts'] = self.map_pts(mp).view(B, self.cfg.n_map, self.cfg.map_pts, 2)
        out['map_cls'] = self.map_cls(mp).squeeze(-1)
        # Planning
        modes = self.mode_embed.weight.unsqueeze(0).expand(B, -1, -1)   # [B,6,D]
        ego_modes = ego + modes                               # ego[B,1,D] broadcast + [B,6,D]
        out['plan_traj'] = self.plan_reg(ego_modes).view(B, self.cfg.fut_mode, self.cfg.fut_ts, 2)
        out['plan_cls']  = self.plan_cls(ego_modes).squeeze(-1)         # [B,6]
        return out

## 8. Full Model (L blocks + Memory + Heads)

Attach heads to each block output and return a list of all block outputs (inference uses final block).

In [ ]:
class DriveTransformerMini(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = FakeSensorEncoder(cfg)
        self.queries = TaskQueries(cfg)
        self.blocks = nn.ModuleList([DriveBlock(cfg) for _ in range(cfg.L)])
        self.heads = Heads(cfg)
    def forward(self, raw_img, ray3d, can_bus, memory):
        img_feats, img_pos = self.encoder(raw_img, ray3d)
        (agent, mp, ego), (apos, mpos, epos) = self.queries(can_bus)
        per_block = []
        for blk in self.blocks:
            agent, mp, ego = blk(agent, mp, ego, apos, mpos, epos, img_feats, img_pos, memory)
            per_block.append(self.heads(agent, mp, ego))
        return per_block, (agent, mp, ego)

model = DriveTransformerMini(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params/1e3:.1f}K')

## 9. Single Frame Forward — Verify Tensor Shapes

In [ ]:
memory = StreamingMemory(cfg)
raw, ray = make_fake_frame(cfg)
can_bus = torch.tensor([[4.0, 0.0]])   # Forward 4 m/s
per_block, (agent, mp, ego) = model(raw, ray, can_bus, memory)
out = per_block[-1]   # Final block
for k, v in out.items():
    print(f'{k:10s} {tuple(v.shape)}')
print('\nBlock count (deep supervision):', len(per_block))

## 10. Streaming Inference — Run Multiple Frames

Per frame: forward → push top queries to FIFO → next frame's temporal attention references the past.
The first frame has an empty queue so temporal is skipped.

In [ ]:
memory.reset()
T = 6
ego_traj_history = []
for t in range(T):
    raw, ray = make_fake_frame(cfg)
    can_bus = torch.tensor([[4.0, 0.3*math.sin(t)]])
    per_block, (agent, mp, ego) = model(raw, ray, can_bus, memory)
    out = per_block[-1]
    # Push top queries to memory
    memory.push(agent, mp, ego, out['det_cls'], out['map_cls'])
    best_mode = out['plan_cls'].argmax(dim=1)[0].item()
    ego_traj_history.append(out['plan_traj'][0, best_mode].detach().numpy())
    qlen = len(memory.buf['agent'])
    print(f't={t}: memory frames={qlen}, best plan mode={best_mode}')
print('\nFIFO is kept at max', cfg.memory_len, 'frames per frame')

## 11. Training Step — Planning WTA Loss and Verifying Gradient Reaches Backbone

The core claim of DriveTransformer is that **planning gradients reach the raw sensor backbone directly, without going through intermediate representations**.
Here we run one backward pass with Winner-Take-All loss against a pseudo-GT trajectory and verify that gradients appear in `encoder.patch_proj` (= backbone equivalent).

In [ ]:
def planning_wta_loss(out, gt_traj):
    # out['plan_traj']: [B,6,fut_ts,2], gt_traj: [B,fut_ts,2]
    err = ((out['plan_traj'] - gt_traj.unsqueeze(1))**2).mean(dim=(2,3))  # [B,6]
    best = err.argmin(dim=1)                                              # Winner mode
    reg = err.gather(1, best.unsqueeze(1)).mean()                        # Regress only winner
    cls = F.cross_entropy(out['plan_cls'], best)                         # Classify to predict winner
    return reg + cls

model.zero_grad()
memory.reset()
raw, ray = make_fake_frame(cfg)
can_bus = torch.tensor([[4.0, 0.0]])
per_block, _ = model(raw, ray, can_bus, memory)
# Pseudo-GT going straight (constant speed forward)
gt = torch.zeros(1, cfg.fut_ts, 2); gt[..., 0] = torch.arange(1, cfg.fut_ts+1).float()
# deep supervision: loss on all blocks
loss = sum(planning_wta_loss(o, gt) for o in per_block) / len(per_block)
loss.backward()
g = model.encoder.patch_proj.weight.grad
print(f'planning loss = {loss.item():.4f}')
print(f'backbone(patch_proj) grad norm = {g.norm().item():.4e}  -> planning gradient reaches raw sensor encoding')
assert g.norm().item() > 0, 'No gradient flowing to backbone!'
print('OK: gradient flowing end-to-end')

## 12. Visualization ① — 6-Mode Planning Trajectories

6 candidate trajectories from ego query + 6 mode embeddings. WTA selects the one closest to GT.

In [ ]:
with torch.no_grad():
    memory.reset()
    raw, ray = make_fake_frame(cfg)
    per_block, _ = model(raw, ray, torch.tensor([[4.0,0.0]]), memory)
    traj = per_block[-1]['plan_traj'][0].numpy()    # [6,fut_ts,2]
    cls  = per_block[-1]['plan_cls'][0].softmax(-1).numpy()
best = cls.argmax()
plt.figure(figsize=(5,5))
for m in range(cfg.fut_mode):
    xy = traj[m]
    lw = 3 if m==best else 1
    plt.plot(xy[:,0], xy[:,1], '-o', lw=lw, label=f'mode{m} p={cls[m]:.2f}'+(' *' if m==best else ''))
plt.scatter([0],[0], c='k', marker='s', s=80, label='ego now')
plt.title('6-mode planning trajectories (bold = selected)'); plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.legend(fontsize=7); plt.axis('equal'); plt.grid(alpha=0.3); plt.show()

## 13. Visualization ② — Task Self-Attention Matrix (Essence of Task Parallelism)

Self-attention weights of `[agent(20) ; map(10) ; ego(1)]`. **Off-diagonal blocks** (agent↔map, ego↔agent)
represent inter-task information exchange — this is the true nature of "task parallelism that eliminates the sequential pipeline".

In [ ]:
# Task self-attn weights of final block (head average)
w = model.blocks[-1]._w_self[0].detach().numpy()   # [Nq,Nq]
na, nm, ne = cfg.n_agent, cfg.n_map, cfg.n_ego
plt.figure(figsize=(6,5))
plt.imshow(w, aspect='auto', cmap='viridis')
for b in [na, na+nm]:
    plt.axhline(b-0.5, color='r', lw=1); plt.axvline(b-0.5, color='r', lw=1)
plt.title('Task Self-Attention weights\n(red lines split agent | map | ego)')
plt.xlabel('key (attended)'); plt.ylabel('query')
plt.colorbar(); plt.tight_layout(); plt.show()
print('Bottom-right small block = ego. Attention from ego row to agent/map columns = interaction of planning reading surroundings')

## Summary

- **Task parallelism**: Put `[agent;map;ego]` into one sequence and apply self-attention. Eliminates stages; all tasks interact at every block (off-diagonal blocks in §13).
- **Sparse representation**: No BEV construction; queries directly cross-attend to image tokens (§6 ②).
- **Streaming**: Past queries stored in FIFO for temporal cross-attention (§5, §10). Far lighter than accumulating dense BEV.
- **6-mode planning + WTA**: Preserve multi-modal driving intent while regressing to the best (§11, §12).
- **End-to-end**: Planning gradients flow directly to backbone without intermediate representations (assert in §11).

Differences from the real model: ResNet backbone, PETR 3D ray geometry, mmdet3d Hungarian matching,
coordinate-transform-aware temporal attention, adaptive LayerNorm temporal conditioning, etc. are simplified.
For the full implementation see the [official repository](https://github.com/Thinklab-SJTU/DriveTransformer).